# Comprehensive Network Input Diagnostic

This notebook tests ALL input pathways to the DeepCFR network:

1. **Hand Card Embeddings** - Do different hole cards produce different features?
2. **Board Card Embeddings** - Do different community cards produce different features?
3. **Action History Encoding** - Do different betting histories produce different features?
4. **Gradient Flow** - Do gradients flow back through ALL branches?
5. **Combined Sensitivity** - Does the network respond to all input types?

**Run all cells and look for FAIL/WARN messages.**

## Setup

In [1]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

# Add paths
sys.path.insert(0, '.')
sys.path.insert(0, '..')

from network.card_embedding import CardEmbedding
from network.model import DeepCFRModule, normalize
from utils.infoset_parser import parse_infoset_to_network_input

# Optional: Load a trained model for comparison
MODEL_PATH = "../output/models/action_encoding_test.pt"  # Change to your model

print("Setup complete!")
print(f"PyTorch version: {torch.__version__}")

Setup complete!
PyTorch version: 2.9.1


---
## Test 1: Hand Card Embedding Differentiation

Do different hole cards produce different embedding vectors?

In [2]:
print("="*70)
print("TEST 1: Hand Card Embedding Differentiation")
print("="*70)

# Create fresh embedding layer
embedding = CardEmbedding(dim=256)

# Card encoding: rank * 10 + suit
# Ace of spades = 14*10 + 0 = 140
# 2 of spades = 2*10 + 0 = 20

test_cards = [
    ("Ace of spades", 140),
    ("2 of spades", 20),
    ("King of hearts", 131),
    ("7 of diamonds", 72),
    ("Ace of hearts", 141),  # Same rank as Ace of spades
]

embeddings = {}
print("\nCard embeddings:")
print("-" * 60)
for name, code in test_cards:
    card_tensor = torch.tensor([[code]])
    emb = embedding(card_tensor)
    embeddings[name] = emb
    print(f"{name} (code={code}): mean={emb.mean().item():.4f}, std={emb.std().item():.4f}")

print("\nPairwise differences:")
print("-" * 40)

diff_ace_2 = (embeddings["Ace of spades"] - embeddings["2 of spades"]).abs().mean().item()
diff_ace_suits = (embeddings["Ace of spades"] - embeddings["Ace of hearts"]).abs().mean().item()
diff_king_7 = (embeddings["King of hearts"] - embeddings["7 of diamonds"]).abs().mean().item()

print(f"Ace vs 2 (different ranks): {diff_ace_2:.6f}")
print(f"As vs Ah (same rank, diff suit): {diff_ace_suits:.6f}")
print(f"King vs 7: {diff_king_7:.6f}")

print("\n" + "="*70)
if diff_ace_2 > 0.01:
    print("PASS: Hand card embeddings differentiate between ranks")
else:
    print("FAIL: Hand card embeddings NOT differentiating!")

TEST 1: Hand Card Embedding Differentiation

Card embeddings:
------------------------------------------------------------
Ace of spades (code=140): mean=-0.1107, std=1.7467
2 of spades (code=20): mean=-0.0119, std=1.7121
King of hearts (code=131): mean=0.0585, std=1.7413
7 of diamonds (code=72): mean=-0.1028, std=1.7579
Ace of hearts (code=141): mean=-0.1412, std=1.7765

Pairwise differences:
----------------------------------------
Ace vs 2 (different ranks): 1.625966
As vs Ah (same rank, diff suit): 1.503348
King vs 7: 2.015678

PASS: Hand card embeddings differentiate between ranks


---
## Test 2: Board Card Embedding Differentiation

Do different community cards produce different embedding vectors?

In [3]:
print("="*70)
print("TEST 2: Board Card Embedding Differentiation")
print("="*70)

# Create fresh network
network = DeepCFRModule(
    nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256
)
network.eval()

# Test with different boards (same hand, same action)
test_boards = [
    ("Dry board (A-K-7)", "S1|H:14s0,10s1,5s2|B:14s3,13s0,7s1|A:"),
    ("Wet board (9-8-7)", "S1|H:14s0,10s1,5s2|B:9s3,8s0,7s1|A:"),  # Connected
    ("Paired board (A-A-7)", "S1|H:14s0,10s1,5s2|B:14s3,14s2,7s1|A:"),  # Paired
    ("Low board (4-3-2)", "S1|H:14s0,10s1,5s2|B:4s3,3s0,2s1|A:"),
]

outputs = {}
print("\nNetwork outputs for same hand, different boards:")
print("-" * 70)

for desc, infoset in test_boards:
    cc, ah = parse_infoset_to_network_input(infoset)
    with torch.no_grad():
        out = network(cc, ah)
    outputs[desc] = out[0]
    print(f"{desc}:")
    print(f"  Logits: [{', '.join([f'{x:.4f}' for x in out[0].tolist()])}]")

print("\nPairwise output differences:")
print("-" * 40)

diff_dry_wet = (outputs["Dry board (A-K-7)"] - outputs["Wet board (9-8-7)"]).abs().mean().item()
diff_dry_paired = (outputs["Dry board (A-K-7)"] - outputs["Paired board (A-A-7)"]).abs().mean().item()
diff_dry_low = (outputs["Dry board (A-K-7)"] - outputs["Low board (4-3-2)"]).abs().mean().item()

print(f"Dry vs Wet board: {diff_dry_wet:.6f}")
print(f"Dry vs Paired board: {diff_dry_paired:.6f}")
print(f"Dry vs Low board: {diff_dry_low:.6f}")

print("\n" + "="*70)
if diff_dry_wet > 0.001 and diff_dry_low > 0.001:
    print("PASS: Board card embeddings differentiate between boards")
else:
    print("FAIL: Board card embeddings NOT differentiating!")

TEST 2: Board Card Embedding Differentiation

Network outputs for same hand, different boards:
----------------------------------------------------------------------
Dry board (A-K-7):
  Logits: [-0.0140, 0.0110, -0.0063, 0.0187, 0.0164, 0.0122, 0.0121, 0.0168, -0.0016]
Wet board (9-8-7):
  Logits: [-0.0131, 0.0141, -0.0051, 0.0171, 0.0088, 0.0152, 0.0100, 0.0140, -0.0032]
Paired board (A-A-7):
  Logits: [-0.0170, 0.0126, -0.0103, 0.0181, 0.0146, 0.0102, 0.0151, 0.0196, -0.0021]
Low board (4-3-2):
  Logits: [-0.0214, 0.0157, -0.0039, 0.0211, 0.0144, 0.0068, 0.0127, 0.0232, 0.0013]

Pairwise output differences:
----------------------------------------
Dry vs Wet board: 0.002654
Dry vs Paired board: 0.002119
Dry vs Low board: 0.003787

PASS: Board card embeddings differentiate between boards


---
## Test 3: Action History Encoding Differentiation

Do different betting histories produce different encodings?

**Important:** We now have 7 features per action: check, call, fold, discard, raise_small, raise_medium, raise_large

In [4]:
print("="*70)
print("TEST 3: Action History Encoding Differentiation")
print("="*70)

# Create fresh network
network = DeepCFRModule(
    nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256
)

# Test different action sequences
test_actions = [
    ("Check-Check", "XX"),
    ("Raise Small-Call", "rC"),
    ("Raise Medium-Call", "RC"),
    ("Raise Large-Call", "BC"),  # Now should be different from RC!
    ("Check-Raise-Fold", "XrF"),
    ("Aggressive: Raise-Raise-Raise", "rRB"),
]

encodings = {}
print("\nAction history encodings:")
print("-" * 70)
print("(Showing first 21 features = first 3 actions * 7 features each)")
print()

for desc, action_str in test_actions:
    enc = network._encode_action_history(action_str)
    encodings[desc] = enc
    # Show first 21 features (3 actions * 7 features)
    first_21 = [f"{x:.0f}" for x in enc[:21].tolist()]
    print(f"{desc} ('{action_str}'):")
    print(f"  Action 1: {first_21[0:7]}")
    print(f"  Action 2: {first_21[7:14]}")
    print(f"  Action 3: {first_21[14:21]}")
    print()

print("Pairwise encoding differences:")
print("-" * 40)

# KEY TEST: Small vs Medium vs Large raises should ALL be different
diff_small_medium = (encodings["Raise Small-Call"] - encodings["Raise Medium-Call"]).abs().sum().item()
diff_medium_large = (encodings["Raise Medium-Call"] - encodings["Raise Large-Call"]).abs().sum().item()
diff_small_large = (encodings["Raise Small-Call"] - encodings["Raise Large-Call"]).abs().sum().item()
diff_check_raise = (encodings["Check-Check"] - encodings["Raise Small-Call"]).abs().sum().item()

print(f"Raise SMALL vs MEDIUM: {diff_small_medium:.1f} (should be > 0)")
print(f"Raise MEDIUM vs LARGE: {diff_medium_large:.1f} (should be > 0)")
print(f"Raise SMALL vs LARGE: {diff_small_large:.1f} (should be > 0)")
print(f"Check vs Raise: {diff_check_raise:.1f}")

print("\n" + "="*70)
if diff_small_medium > 0 and diff_medium_large > 0:
    print("PASS: All 3 raise sizes are distinguishable in encoding!")
else:
    print("FAIL: Raise sizes NOT distinguishable - check action_map!")
    print("      Expected: 'r'->idx 4, 'R'->idx 5, 'B'->idx 6")

TEST 3: Action History Encoding Differentiation

Action history encodings:
----------------------------------------------------------------------
(Showing first 21 features = first 3 actions * 7 features each)

Check-Check ('XX'):
  Action 1: ['1', '0', '0', '0', '0', '0', '0']
  Action 2: ['1', '0', '0', '0', '0', '0', '0']
  Action 3: ['0', '0', '0', '0', '0', '0', '0']

Raise Small-Call ('rC'):
  Action 1: ['0', '0', '0', '0', '1', '0', '0']
  Action 2: ['0', '1', '0', '0', '0', '0', '0']
  Action 3: ['0', '0', '0', '0', '0', '0', '0']

Raise Medium-Call ('RC'):
  Action 1: ['0', '0', '0', '0', '0', '1', '0']
  Action 2: ['0', '1', '0', '0', '0', '0', '0']
  Action 3: ['0', '0', '0', '0', '0', '0', '0']

Raise Large-Call ('BC'):
  Action 1: ['0', '0', '0', '0', '0', '0', '1']
  Action 2: ['0', '1', '0', '0', '0', '0', '0']
  Action 3: ['0', '0', '0', '0', '0', '0', '0']

Check-Raise-Fold ('XrF'):
  Action 1: ['1', '0', '0', '0', '0', '0', '0']
  Action 2: ['0', '0', '0', '0', '1', '

---
## Test 4: Network Response to Action History

Does the network produce different outputs for different action histories?

In [5]:
print("="*70)
print("TEST 4: Network Response to Action History")
print("="*70)

# Create fresh network
network = DeepCFRModule(
    nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256
)
network.eval()

# Same hand and board, different action histories
base_hand = "14s0,10s1,5s2"  # A-T-5
base_board = "9s3,8s0,7s1"   # 9-8-7 flop

test_scenarios = [
    ("No action", f"S1|H:{base_hand}|B:{base_board}|A:"),
    ("After check", f"S1|H:{base_hand}|B:{base_board}|A:X"),
    ("After small raise", f"S1|H:{base_hand}|B:{base_board}|A:r"),
    ("After medium raise", f"S1|H:{base_hand}|B:{base_board}|A:R"),
    ("After large raise", f"S1|H:{base_hand}|B:{base_board}|A:B"),
    ("Aggressive history", f"S1|H:{base_hand}|B:{base_board}|A:rRBC"),
]

outputs = {}
print("\nNetwork outputs for same hand/board, different histories:")
print("-" * 70)

for desc, infoset in test_scenarios:
    cc, ah = parse_infoset_to_network_input(infoset)
    with torch.no_grad():
        out = network(cc, ah)
    outputs[desc] = out[0]
    print(f"{desc}:")
    print(f"  Logits: [{', '.join([f'{x:.4f}' for x in out[0].tolist()])}]")

print("\nPairwise output differences:")
print("-" * 40)

diff_none_check = (outputs["No action"] - outputs["After check"]).abs().mean().item()
diff_small_medium = (outputs["After small raise"] - outputs["After medium raise"]).abs().mean().item()
diff_medium_large = (outputs["After medium raise"] - outputs["After large raise"]).abs().mean().item()
diff_check_aggressive = (outputs["After check"] - outputs["Aggressive history"]).abs().mean().item()

print(f"No action vs After check: {diff_none_check:.6f}")
print(f"After SMALL vs MEDIUM raise: {diff_small_medium:.6f}")
print(f"After MEDIUM vs LARGE raise: {diff_medium_large:.6f}")
print(f"After check vs Aggressive: {diff_check_aggressive:.6f}")

print("\n" + "="*70)
if diff_small_medium > 0.0001 and diff_medium_large > 0.0001:
    print("PASS: Network distinguishes all raise sizes in history!")
else:
    print("FAIL: Network NOT distinguishing raise sizes!")

TEST 4: Network Response to Action History

Network outputs for same hand/board, different histories:
----------------------------------------------------------------------
No action:
  Logits: [0.0020, -0.0052, 0.0016, 0.0122, -0.0228, -0.0031, -0.0134, -0.0093, -0.0130]
After check:
  Logits: [0.0007, -0.0071, 0.0028, 0.0108, -0.0206, -0.0024, -0.0140, -0.0139, -0.0103]
After small raise:
  Logits: [0.0022, -0.0037, 0.0047, 0.0113, -0.0246, -0.0034, -0.0143, -0.0051, -0.0160]
After medium raise:
  Logits: [0.0057, -0.0049, 0.0039, 0.0141, -0.0250, -0.0072, -0.0144, -0.0050, -0.0127]
After large raise:
  Logits: [0.0014, -0.0091, 0.0039, 0.0108, -0.0262, -0.0046, -0.0167, -0.0101, -0.0136]
Aggressive history:
  Logits: [0.0038, -0.0010, 0.0009, 0.0139, -0.0301, -0.0068, -0.0103, -0.0188, -0.0073]

Pairwise output differences:
----------------------------------------
No action vs After check: 0.001832
After SMALL vs MEDIUM raise: 0.001756
After MEDIUM vs LARGE raise: 0.002654
After che

---
## Test 5: Gradient Flow Through All Branches

Do gradients flow back to:
- Hand card embeddings
- Board card embeddings  
- Action history layers

**This is critical!** If gradients don't flow, the network can't learn from those inputs.

In [6]:
print("="*70)
print("TEST 5: Gradient Flow Through All Branches")
print("="*70)

# Create network for gradient testing
grad_net = DeepCFRModule(
    nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256
)
grad_net.train()

# Forward pass with a post-flop scenario (has board cards)
cc, ah = parse_infoset_to_network_input("S1|H:14s0,10s1,5s2|B:9s3,8s0,7s1|A:rC")
output = grad_net(cc, ah)

# Create loss and backpropagate
target = torch.zeros_like(output)
target[0, 6] = 1.0  # Target: RAISE_SMALL

loss = F.mse_loss(output, target)
loss.backward()

# Check gradients at ALL key layers
print("\nGradient magnitudes at all key layers:")
print("-" * 80)

layers_to_check = [
    # Hand branch
    ('HAND: embeddings[0].card', grad_net.hand_embeddings[0].card.weight),
    ('HAND: embeddings[0].rank', grad_net.hand_embeddings[0].rank.weight),
    ('HAND: embeddings[0].suit', grad_net.hand_embeddings[0].suit.weight),
    ('HAND: layer1.weight', grad_net.hand_layer1.weight),
    ('HAND: layer2.weight', grad_net.hand_layer2.weight),
    ('HAND: layer3.weight', grad_net.hand_layer3.weight),
    
    # Board branch
    ('BOARD: embeddings[0].card', grad_net.board_embeddings[0].card.weight),
    ('BOARD: embeddings[0].rank', grad_net.board_embeddings[0].rank.weight),
    ('BOARD: layer1.weight', grad_net.board_layer1.weight),
    ('BOARD: layer2.weight', grad_net.board_layer2.weight),
    ('BOARD: layer3.weight', grad_net.board_layer3.weight),
    
    # Action branch
    ('ACTION: layer1.weight', grad_net.actions_layer1.weight),
    ('ACTION: layer2.weight', grad_net.actions_layer2.weight),
    
    # Combined trunk
    ('COMBINED: layer1.weight', grad_net.comb_layer1.weight),
    ('COMBINED: layer2.weight', grad_net.comb_layer2.weight),
    ('COMBINED: layer3.weight', grad_net.comb_layer3.weight),
    
    # Output
    ('OUTPUT: action_head.weight', grad_net.action_head.weight),
    ('OUTPUT: action_head.bias', grad_net.action_head.bias),
]

print(f"{'Layer':<35} {'Grad Mean':<15} {'Grad Max':<15} {'Status'}")
print("-" * 80)

results = {'hand': [], 'board': [], 'action': [], 'combined': [], 'output': []}

for name, param in layers_to_check:
    if param.grad is not None:
        grad_mean = param.grad.abs().mean().item()
        grad_max = param.grad.abs().max().item()
        if grad_mean > 1e-10:
            status = "OK"
        else:
            status = "ZERO!"
    else:
        grad_mean = 0
        grad_max = 0
        status = "NO GRAD!"
    
    print(f"{name:<35} {grad_mean:<15.2e} {grad_max:<15.2e} {status}")
    
    # Categorize
    if 'HAND' in name:
        results['hand'].append(grad_mean > 1e-10)
    elif 'BOARD' in name:
        results['board'].append(grad_mean > 1e-10)
    elif 'ACTION' in name:
        results['action'].append(grad_mean > 1e-10)
    elif 'COMBINED' in name:
        results['combined'].append(grad_mean > 1e-10)
    else:
        results['output'].append(grad_mean > 1e-10)

print("\n" + "="*70)
print("SUMMARY BY BRANCH:")
print("-" * 40)

all_ok = True
for branch, grads_ok in results.items():
    if all(grads_ok):
        print(f"  {branch.upper():10s}: PASS - gradients flowing")
    else:
        print(f"  {branch.upper():10s}: FAIL - some layers have zero gradients!")
        all_ok = False

print("\n" + "="*70)
if all_ok:
    print("PASS: Gradients flow through ALL branches!")
else:
    print("FAIL: Some branches have blocked gradients!")
    print("      Check for zero-weight initialization or detach() calls.")

TEST 5: Gradient Flow Through All Branches

Gradient magnitudes at all key layers:
--------------------------------------------------------------------------------
Layer                               Grad Mean       Grad Max        Status
--------------------------------------------------------------------------------
HAND: embeddings[0].card            1.91e-07        1.18e-04        OK
HAND: embeddings[0].rank            1.91e-06        1.18e-04        OK
HAND: embeddings[0].suit            7.16e-06        1.18e-04        OK
HAND: layer1.weight                 8.17e-05        2.40e-03        OK
HAND: layer2.weight                 5.51e-05        3.30e-03        OK
HAND: layer3.weight                 5.77e-05        3.07e-03        OK
BOARD: embeddings[0].card           1.43e-07        6.99e-05        OK
BOARD: embeddings[0].rank           1.43e-06        6.99e-05        OK
BOARD: layer1.weight                4.83e-05        2.74e-03        OK
BOARD: layer2.weight                4.77e

---
## Test 6: Feature Flow Through Layers (per branch)

Track how input differences propagate through each branch.

In [7]:
print("="*70)
print("TEST 6: Feature Flow Through Layers")
print("="*70)

# Create fresh network
test_net = DeepCFRModule(
    nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256
)
test_net.eval()

# Storage for activations
activations = {}

def make_hook(name):
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

# Register hooks
hooks = [
    test_net.hand_layer1.register_forward_hook(make_hook('hand_l1')),
    test_net.hand_layer2.register_forward_hook(make_hook('hand_l2')),
    test_net.hand_layer3.register_forward_hook(make_hook('hand_l3')),
    test_net.board_layer1.register_forward_hook(make_hook('board_l1')),
    test_net.board_layer2.register_forward_hook(make_hook('board_l2')),
    test_net.board_layer3.register_forward_hook(make_hook('board_l3')),
    test_net.actions_layer1.register_forward_hook(make_hook('action_l1')),
    test_net.actions_layer2.register_forward_hook(make_hook('action_l2')),
    test_net.comb_layer1.register_forward_hook(make_hook('comb_l1')),
    test_net.comb_layer2.register_forward_hook(make_hook('comb_l2')),
    test_net.comb_layer3.register_forward_hook(make_hook('comb_l3')),
    test_net.action_head.register_forward_hook(make_hook('output')),
]

# Compare different scenarios
scenarios = [
    # Different hands (same board, same action)
    ("Hand: AA vs 23",
     "S1|H:14s0,14s1,5s2|B:9s3,8s0,7s1|A:rC",
     "S1|H:3s0,2s1,5s2|B:9s3,8s0,7s1|A:rC"),
    
    # Different boards (same hand, same action)
    ("Board: High vs Low",
     "S1|H:14s0,10s1,5s2|B:14s3,13s0,12s1|A:rC",
     "S1|H:14s0,10s1,5s2|B:4s3,3s0,2s1|A:rC"),
    
    # Different actions (same hand, same board)
    ("Action: Check vs Big Raise",
     "S1|H:14s0,10s1,5s2|B:9s3,8s0,7s1|A:XX",
     "S1|H:14s0,10s1,5s2|B:9s3,8s0,7s1|A:BC"),
]

for scenario_name, infoset1, infoset2 in scenarios:
    print(f"\n{scenario_name}:")
    print("-" * 70)
    
    cc1, ah1 = parse_infoset_to_network_input(infoset1)
    cc2, ah2 = parse_infoset_to_network_input(infoset2)
    
    with torch.no_grad():
        _ = test_net(cc1, ah1)
        act1 = {k: v.clone() for k, v in activations.items()}
        
        _ = test_net(cc2, ah2)
        act2 = {k: v.clone() for k, v in activations.items()}
    
    print(f"{'Layer':<15} {'Mean Diff':<12} {'Where expected'}")
    
    for layer in ['hand_l1', 'hand_l2', 'hand_l3', 
                  'board_l1', 'board_l2', 'board_l3',
                  'action_l1', 'action_l2',
                  'comb_l1', 'comb_l2', 'comb_l3', 'output']:
        diff = (act1[layer] - act2[layer]).abs().mean().item()
        
        # Determine expected behavior
        if 'Hand' in scenario_name:
            expected = 'hand' in layer or 'comb' in layer or 'output' in layer
        elif 'Board' in scenario_name:
            expected = 'board' in layer or 'comb' in layer or 'output' in layer
        else:  # Action
            expected = 'action' in layer or 'comb' in layer or 'output' in layer
        
        marker = "<--" if expected and diff > 0.001 else "" if not expected else "(expected)"
        print(f"{layer:<15} {diff:<12.6f} {marker}")

# Clean up hooks
for h in hooks:
    h.remove()

TEST 6: Feature Flow Through Layers

Hand: AA vs 23:
----------------------------------------------------------------------
Layer           Mean Diff    Where expected
hand_l1         1.163892     <--
hand_l2         0.437932     <--
hand_l3         0.144094     <--
board_l1        0.000000     
board_l2        0.000000     
board_l3        0.000000     
action_l1       0.000000     
action_l2       0.000000     
comb_l1         0.030401     <--
comb_l2         0.013236     <--
comb_l3         0.013664     <--
output          0.004876     <--

Board: High vs Low:
----------------------------------------------------------------------
Layer           Mean Diff    Where expected
hand_l1         0.000000     
hand_l2         0.000000     
hand_l3         0.000000     
board_l1        0.640561     <--
board_l2        0.225046     <--
board_l3        0.081512     <--
action_l1       0.000000     
action_l2       0.000000     
comb_l1         0.017042     <--
comb_l2         0.006787     <--


---
## Test 7: Combined Input Sensitivity

Test that the network is sensitive to ALL inputs simultaneously.

In [8]:
print("="*70)
print("TEST 7: Combined Input Sensitivity")
print("="*70)

network = DeepCFRModule(
    nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256
)
network.eval()

# Base scenario
base = "S1|H:10s0,9s1,5s2|B:8s3,7s0,6s1|A:rC"

# Variations changing one thing at a time
variations = [
    ("Base", base),
    ("Better hand", "S1|H:14s0,14s1,5s2|B:8s3,7s0,6s1|A:rC"),
    ("Worse hand", "S1|H:4s0,3s1,2s2|B:8s3,7s0,6s1|A:rC"),
    ("Better board", "S1|H:10s0,9s1,5s2|B:14s3,13s0,12s1|A:rC"),
    ("Worse board", "S1|H:10s0,9s1,5s2|B:4s3,3s0,2s1|A:rC"),
    ("Passive action", "S1|H:10s0,9s1,5s2|B:8s3,7s0,6s1|A:XX"),
    ("Aggressive action", "S1|H:10s0,9s1,5s2|B:8s3,7s0,6s1|A:BBC"),
]

outputs = {}
print("\nOutputs for each variation:")
print("-" * 70)

for name, infoset in variations:
    cc, ah = parse_infoset_to_network_input(infoset)
    with torch.no_grad():
        out = network(cc, ah)
    outputs[name] = out[0]
    print(f"{name:20s}: {[f'{x:.4f}' for x in out[0].tolist()]}")

print("\nSensitivity analysis (difference from base):")
print("-" * 50)

base_out = outputs["Base"]
sensitivities = {}

for name, out in outputs.items():
    if name != "Base":
        diff = (out - base_out).abs().mean().item()
        sensitivities[name] = diff
        print(f"{name:20s}: {diff:.6f}")

print("\n" + "="*70)

# Check that all input types cause changes
hand_sensitive = (sensitivities["Better hand"] > 0.001 or sensitivities["Worse hand"] > 0.001)
board_sensitive = (sensitivities["Better board"] > 0.001 or sensitivities["Worse board"] > 0.001)
action_sensitive = (sensitivities["Passive action"] > 0.001 or sensitivities["Aggressive action"] > 0.001)

print(f"Hand sensitivity:   {'PASS' if hand_sensitive else 'FAIL'}")
print(f"Board sensitivity:  {'PASS' if board_sensitive else 'FAIL'}")
print(f"Action sensitivity: {'PASS' if action_sensitive else 'FAIL'}")

if hand_sensitive and board_sensitive and action_sensitive:
    print("\nPASS: Network is sensitive to ALL input types!")
else:
    print("\nFAIL: Network is NOT sensitive to some inputs!")

TEST 7: Combined Input Sensitivity

Outputs for each variation:
----------------------------------------------------------------------
Base                : ['-0.0101', '0.0024', '0.0312', '0.0125', '-0.0026', '0.0187', '-0.0075', '-0.0146', '-0.0138']
Better hand         : ['-0.0063', '0.0037', '0.0248', '0.0142', '-0.0055', '0.0179', '-0.0098', '-0.0186', '-0.0107']
Worse hand          : ['-0.0122', '-0.0019', '0.0246', '0.0032', '-0.0177', '0.0138', '-0.0039', '-0.0063', '-0.0191']
Better board        : ['-0.0057', '0.0026', '0.0230', '0.0132', '-0.0070', '0.0147', '-0.0080', '-0.0076', '-0.0066']
Worse board         : ['-0.0128', '0.0025', '0.0224', '0.0159', '0.0041', '0.0165', '-0.0047', '-0.0057', '-0.0101']
Passive action      : ['-0.0063', '0.0055', '0.0269', '0.0098', '0.0008', '0.0140', '-0.0110', '-0.0165', '-0.0144']
Aggressive action   : ['-0.0076', '0.0120', '0.0267', '0.0108', '-0.0039', '0.0152', '-0.0113', '-0.0179', '-0.0095']

Sensitivity analysis (difference from b

---
## Test 8: Trained Model Comparison

Compare fresh vs trained model sensitivity.

In [9]:
print("="*70)
print("TEST 8: Trained Model Comparison")
print("="*70)

# Load trained model
try:
    model_data = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
    network_dim = model_data.get('network_dim', 256)
    iterations = model_data.get('iterations', '?')
    
    trained_net = DeepCFRModule(
        nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=network_dim
    )
    trained_net.load_state_dict(model_data['strategy_network_state_dict'])
    trained_net.eval()
    print(f"Loaded: {MODEL_PATH} ({iterations} iterations)")
    
    fresh_net = DeepCFRModule(
        nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=network_dim
    )
    fresh_net.eval()
    
    # Test scenarios
    test_cases = [
        ("Premium hand (AA)", "S0|H:14s0,14s1,5s2|B:|A:"),
        ("Garbage hand (23)", "S0|H:3s0,2s1,4s2|B:|A:"),
        ("After big raise", "S1|H:14s0,14s1,5s2|B:9s3,8s0,7s1|A:BC"),
    ]
    
    print("\nComparison of fresh vs trained network:")
    print("-" * 70)
    
    for name, infoset in test_cases:
        cc, ah = parse_infoset_to_network_input(infoset)
        
        with torch.no_grad():
            fresh_out = fresh_net(cc, ah)[0]
            trained_out = trained_net(cc, ah)[0]
        
        print(f"\n{name}:")
        print(f"  Fresh:   {[f'{x:.3f}' for x in fresh_out.tolist()]}")
        print(f"  Trained: {[f'{x:.3f}' for x in trained_out.tolist()]}")
        
        # Show probabilities (softmax)
        fresh_probs = F.softmax(fresh_out, dim=0)
        trained_probs = F.softmax(trained_out, dim=0)
        print(f"  Fresh probs:   {[f'{x:.1%}' for x in fresh_probs.tolist()]}")
        print(f"  Trained probs: {[f'{x:.1%}' for x in trained_probs.tolist()]}")
    
    # Check if trained model differentiates better
    print("\n" + "="*70)
    
    # Compare AA vs 23 differentiation
    cc_aa, ah_aa = parse_infoset_to_network_input("S0|H:14s0,14s1,5s2|B:|A:")
    cc_23, ah_23 = parse_infoset_to_network_input("S0|H:3s0,2s1,4s2|B:|A:")
    
    with torch.no_grad():
        fresh_aa = fresh_net(cc_aa, ah_aa)[0]
        fresh_23 = fresh_net(cc_23, ah_23)[0]
        trained_aa = trained_net(cc_aa, ah_aa)[0]
        trained_23 = trained_net(cc_23, ah_23)[0]
    
    fresh_diff = (fresh_aa - fresh_23).abs().mean().item()
    trained_diff = (trained_aa - trained_23).abs().mean().item()
    
    print(f"AA vs 23 differentiation:")
    print(f"  Fresh model:   {fresh_diff:.6f}")
    print(f"  Trained model: {trained_diff:.6f}")
    
    if trained_diff > fresh_diff:
        print(f"\nPASS: Trained model differentiates better ({trained_diff/fresh_diff:.1f}x)")
    else:
        print(f"\nWARN: Trained model does NOT differentiate better!")
        print("      Network may not be learning from card inputs.")
        
except FileNotFoundError:
    print(f"Model not found: {MODEL_PATH}")
    print("Skipping trained model comparison.")
except Exception as e:
    print(f"Error loading model: {e}")

TEST 8: Trained Model Comparison
Loaded: ../output/models/action_encoding_test.pt (3 iterations)

Comparison of fresh vs trained network:
----------------------------------------------------------------------

Premium hand (AA):
  Fresh:   ['0.009', '0.008', '-0.003', '0.009', '0.008', '0.005', '0.004', '-0.005', '-0.023']
  Trained: ['0.962', '-0.061', '-0.012', '0.163', '0.089', '0.658', '0.053', '-0.009', '0.013']
  Fresh probs:   ['11.2%', '11.2%', '11.1%', '11.2%', '11.2%', '11.2%', '11.1%', '11.0%', '10.8%']
  Trained probs: ['22.2%', '8.0%', '8.4%', '10.0%', '9.3%', '16.4%', '8.9%', '8.4%', '8.6%']

Garbage hand (23):
  Fresh:   ['0.013', '0.013', '-0.006', '0.009', '0.011', '0.004', '-0.002', '-0.010', '-0.019']
  Trained: ['0.942', '-0.066', '-0.006', '0.104', '0.117', '0.657', '0.066', '-0.004', '0.025']
  Fresh probs:   ['11.2%', '11.2%', '11.0%', '11.2%', '11.2%', '11.1%', '11.1%', '11.0%', '10.9%']
  Trained probs: ['21.8%', '8.0%', '8.5%', '9.4%', '9.6%', '16.4%', '9.1%',

---
## Summary

In [10]:
print("="*70)
print("DIAGNOSTIC SUMMARY")
print("="*70)
print()
print("Run all cells above and check for FAIL/WARN messages.")
print()
print("Key things to verify:")
print("-" * 60)
print()
print("1. HAND CARD EMBEDDINGS:")
print("   - Different cards should produce different embeddings")
print("   - Gradients should flow back to embedding layers")
print()
print("2. BOARD CARD EMBEDDINGS:")
print("   - Different boards should produce different outputs")
print("   - Gradients should flow back to board embedding layers")
print()
print("3. ACTION HISTORY ENCODING:")
print("   - All 3 raise sizes (r, R, B) should be distinguishable")
print("   - Encoding should use 7 features per action")
print("   - Gradients should flow back to action layers")
print()
print("4. COMBINED SENSITIVITY:")
print("   - Network should respond to changes in ALL input types")
print("   - Trained model should differentiate hands better than fresh")
print()
print("If any branch shows ZERO gradients or IDENTICAL outputs,")
print("that branch has a bug blocking information flow!")

DIAGNOSTIC SUMMARY

Run all cells above and check for FAIL/WARN messages.

Key things to verify:
------------------------------------------------------------

1. HAND CARD EMBEDDINGS:
   - Different cards should produce different embeddings
   - Gradients should flow back to embedding layers

2. BOARD CARD EMBEDDINGS:
   - Different boards should produce different outputs
   - Gradients should flow back to board embedding layers

3. ACTION HISTORY ENCODING:
   - All 3 raise sizes (r, R, B) should be distinguishable
   - Encoding should use 7 features per action
   - Gradients should flow back to action layers

4. COMBINED SENSITIVITY:
   - Network should respond to changes in ALL input types
   - Trained model should differentiate hands better than fresh

If any branch shows ZERO gradients or IDENTICAL outputs,
that branch has a bug blocking information flow!
